# 02 — Cleaning
Reads raw Delta tables, applies cleaning transformations, and writes to the clean layer.

**Catalog:** `airbnb_app`  
**Reads from:** `airbnb_app.raw`  
**Writes to:** `airbnb_app.clean`  

| Section | Tables | Key transformations |
|---------|--------|--------------------|
| A — Listings | 4 (one per city) | Select 50 cols, cast types, flag outliers, standardise neighbourhood |
| B — Calendar | 4 | Cast date/boolean, drop null rows, drop 100% null price columns |
| C — Reviews | 4 | Drop null comments, trim, add comment_length |
| D — Neighbourhoods | 4 | Standardise casing, drop null names |
| E — Neighbourhoods GeoJSON | 4 | Pass through as-is |
| F — House prices | 1 | Keep Sep 2015 and Sep 2025 only, strip commas, cast to float |
| G — Amenities | 3 | Cast counts to int, handle suppressed [c] values |
| H — Rent data | 1 | Filter to most recent 12 months, keep rental price columns only |

## 0. Config

In [0]:
RAW_DB   = "airbnb_app.raw"
CLEAN_DB = "airbnb_app.clean"

CITIES = ["london", "manchester", "edinburgh", "bristol"]

# IQR multiplier for outlier flagging — rows kept, boolean flag added
IQR_MULTIPLIER = 3.0

# House prices — two periods kept for 10-year price growth calculation
HP_PERIOD_RECENT   = "year_ending_sep_2025"
HP_PERIOD_BASELINE = "year_ending_sep_2015"

# ── Listings columns to keep from 78 ─────────────────────────────────────────
LISTINGS_COLUMNS = [
    # Identity
    "id", "name", "description",
    # Host
    "host_id", "host_name", "host_since",
    "host_response_time", "host_response_rate", "host_acceptance_rate",
    "host_is_superhost", "host_listings_count", "host_identity_verified",
    # Location
    "neighbourhood_cleansed", "latitude", "longitude",
    # Property
    "property_type", "room_type", "accommodates",
    "bathrooms", "bedrooms", "beds", "amenities",
    # Pricing
    "price", "minimum_nights", "maximum_nights",
    # Availability
    "availability_30", "availability_60", "availability_90", "availability_365",
    # Reviews
    "number_of_reviews", "number_of_reviews_ltm",
    "number_of_reviews_l30d", "number_of_reviews_ly",
    "first_review", "last_review",
    "review_scores_rating", "review_scores_accuracy",
    "review_scores_cleanliness", "review_scores_checkin",
    "review_scores_communication", "review_scores_location",
    "review_scores_value", "reviews_per_month",
    # Revenue estimates (Inside Airbnb pre-calculated)
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
    # Regulatory
    "license", "instant_bookable",
    # Metadata
    "_city", "_ingested_at",
]

# ── Calendar columns ──────────────────────────────────────────────────────────
CALENDAR_COLUMNS = [
    "listing_id", "date", "available",
    "minimum_nights", "maximum_nights",
    "_city", "_ingested_at",
    # DROPPED: price, adjusted_price — 100% null
]

# ── Reviews columns ───────────────────────────────────────────────────────────
REVIEWS_COLUMNS = [
    "listing_id", "id", "date",
    "reviewer_id", "reviewer_name", "comments",
    "_city", "_ingested_at",
]

# ── Neighbourhoods columns ────────────────────────────────────────────────────
NEIGHBOURHOODS_COLUMNS = [
    "neighbourhood", "_city", "_ingested_at",
    # DROPPED: neighbourhood_group — 100% null
]

# ── GeoJSON columns ───────────────────────────────────────────────────────────
GEOJSON_COLUMNS = ["geojson_raw", "_city", "_ingested_at"]

## 1. Setup

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import FloatType, IntegerType, BooleanType

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CLEAN_DB}")
print(f"Schema ready: {CLEAN_DB}")

clean_log = []

## 2. Helpers

In [0]:
def write_clean(df: DataFrame, table: str):
    """Write a cleaned DataFrame to the clean Delta layer."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    print(f"  ✓ {table} ({df.count():,} rows)")


def select_available(df: DataFrame, columns: list) -> DataFrame:
    """Select only columns that exist — safe across cities with minor schema differences."""
    available = [c for c in columns if c in df.columns]
    missing   = [c for c in columns if c not in df.columns]
    if missing:
        print(f"  WARNING: columns not found and skipped: {missing}")
    return df.select(available)


def flag_outliers(df: DataFrame, col: str, multiplier: float = IQR_MULTIPLIER) -> DataFrame:
    """
    Flag outliers using IQR method. Adds boolean column {col}_is_outlier.
    Rows are kept — flag allows downstream filtering per use case.
    """
    q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0.01)
    iqr     = q3 - q1
    lower   = q1 - multiplier * iqr
    upper   = q3 + multiplier * iqr
    return df.withColumn(
        f"{col}_is_outlier",
        (F.col(col) < lower) | (F.col(col) > upper)
    )


def cast_tf_boolean(df: DataFrame, col: str) -> DataFrame:
    """Cast Airbnb t/f string columns to proper boolean."""
    return df.withColumn(
        col,
        F.when(F.col(col) == "t", True)
         .when(F.col(col) == "f", False)
         .otherwise(None)
         .cast(BooleanType())
    )


print("Helpers loaded.")

---
# Part A — Listings

**Raw schema:** 78 columns  
**Clean schema:** 50 columns (44 source + outlier flags + `_cleaned_at`)  

| Transformation | Detail |
|---------------|--------|
| Column selection | 44 columns kept from 78 |
| `price` | Strip `$` and `,`, cast to float, drop null/zero |
| `host_response_rate`, `host_acceptance_rate` | Strip `%`, cast to float, divide by 100 |
| `host_is_superhost`, `host_identity_verified`, `instant_bookable` | Cast `t`/`f` → boolean |
| `neighbourhood_cleansed` | `initcap` + `trim` to standardise casing for joins |
| Numeric columns | Cast to int or float |
| Outlier flags | `price_is_outlier`, `minimum_nights_is_outlier` using IQR × 3 |

In [0]:
print("Cleaning listings...")

for city in CITIES:
    src = f"{RAW_DB}.airbnb_listings_{city}"
    tgt = f"{CLEAN_DB}.airbnb_listings_{city}"
    print(f"\n  {city}")

    try:
        df = spark.table(src)
        df = select_available(df, LISTINGS_COLUMNS)
        # Replace N/A strings with null before type casting
        df = df.replace("N/A", None)

        # Price: strip $ and commas, cast to float, drop null/zero
        df = df.withColumn(
            "price",
            F.regexp_replace(F.col("price"), "[\\$,]", "").cast(FloatType())
        )
        df = df.filter(F.col("price").isNotNull() & (F.col("price") > 0))

        # Integer columns
        int_cols = [
            "minimum_nights", "maximum_nights", "accommodates",
            "bedrooms", "beds", "availability_30", "availability_60",
            "availability_90", "availability_365",
            "number_of_reviews", "number_of_reviews_ltm",
            "number_of_reviews_l30d", "number_of_reviews_ly",
            "host_listings_count", "estimated_occupancy_l365d",
            "estimated_revenue_l365d"
        ]
        for c in int_cols:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast(IntegerType()))

        # Float columns
        float_cols = [
            "review_scores_rating", "review_scores_accuracy",
            "review_scores_cleanliness", "review_scores_checkin",
            "review_scores_communication", "review_scores_location",
            "review_scores_value", "reviews_per_month",
            "latitude", "longitude", "bathrooms"
        ]
        for c in float_cols:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast(FloatType()))

        # Host rate columns: strip % and convert to 0-1
        for c in ["host_response_rate", "host_acceptance_rate"]:
            if c in df.columns:
                df = df.withColumn(
                    c,
                    F.regexp_replace(F.col(c), "%", "").cast(FloatType()) / 100
                )

        # Boolean columns
        for c in ["host_is_superhost", "host_identity_verified", "instant_bookable"]:
            if c in df.columns:
                df = cast_tf_boolean(df, c)

        # Standardise neighbourhood casing
        df = df.withColumn(
            "neighbourhood_cleansed",
            F.initcap(F.trim(F.col("neighbourhood_cleansed")))
        )

        # Outlier flags — rows kept, boolean flag added
        df = flag_outliers(df, "price")
        df = flag_outliers(df, "minimum_nights")

        df = df.withColumn("_cleaned_at", F.current_timestamp())
        write_clean(df, tgt)
        clean_log.append({"section": "listings", "city": city, "status": "ok", "rows": df.count()})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        clean_log.append({"section": "listings", "city": city, "status": "error", "error": str(e)})

---
# Part B — Calendar

**Raw schema:** 10 columns  
**Clean schema:** 7 columns  

| Transformation | Detail |
|---------------|--------|
| `available` | Cast `t`/`f` → boolean |
| `price`, `adjusted_price` | Dropped — 100% null |
| Null rows | Drop rows with null `listing_id` or `date` |

> **Limitation:** `available = false` cannot distinguish booked from host-blocked nights. Occupancy proxies must state this assumption.

In [0]:
print("Cleaning calendar...")

for city in CITIES:
    src = f"{RAW_DB}.airbnb_calendar_{city}"
    tgt = f"{CLEAN_DB}.airbnb_calendar_{city}"
    print(f"\n  {city}")

    try:
        df = spark.table(src)
        df = select_available(df, CALENDAR_COLUMNS)
        df = cast_tf_boolean(df, "available")
        df = df.replace("N/A", None)  

        for c in ["minimum_nights", "maximum_nights"]:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast(IntegerType()))

        df = df.filter(
            F.col("listing_id").isNotNull() &
            F.col("date").isNotNull()
        )

        df = df.withColumn("_cleaned_at", F.current_timestamp())
        write_clean(df, tgt)
        clean_log.append({"section": "calendar", "city": city, "status": "ok", "rows": df.count()})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        clean_log.append({"section": "calendar", "city": city, "status": "error", "error": str(e)})

---
# Part C — Reviews

**Raw schema:** 9 columns  
**Clean schema:** 9 columns + `comment_length`  

| Transformation | Detail |
|---------------|--------|
| Null comments | Dropped |
| `comments` | Trimmed of whitespace |
| `comment_length` | Added — used to filter very short reviews before AI analysis |
| IDs | Cast to long |

> Reviews may contain non-English comments — handle in the AI notebook.

In [0]:
print("Cleaning reviews...")

for city in CITIES:
    src = f"{RAW_DB}.airbnb_reviews_{city}"
    tgt = f"{CLEAN_DB}.airbnb_reviews_{city}"
    print(f"\n  {city}")

    try:
        df = spark.table(src)
        df = select_available(df, REVIEWS_COLUMNS)

        # Drop null or empty comments
        df = df.filter(
            F.col("comments").isNotNull() &
            (F.length(F.trim(F.col("comments"))) > 0)
        )

        df = df.withColumn("comments", F.trim(F.col("comments")))
        df = df.withColumn("comment_length", F.length(F.col("comments")))

        for c in ["listing_id", "id", "reviewer_id"]:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast("long"))

        df = df.withColumn("_cleaned_at", F.current_timestamp())
        write_clean(df, tgt)
        clean_log.append({"section": "reviews", "city": city, "status": "ok", "rows": df.count()})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        clean_log.append({"section": "reviews", "city": city, "status": "error", "error": str(e)})

---
# Part D — Neighbourhoods

**Raw schema:** 5 columns  
**Clean schema:** 3 columns  

| Transformation | Detail |
|---------------|--------|
| `neighbourhood_group` | Dropped — 100% null |
| `neighbourhood` | `initcap` + `trim` to match `neighbourhood_cleansed` in listings |

In [0]:
print("Cleaning neighbourhoods...")

for city in CITIES:
    src = f"{RAW_DB}.airbnb_neighbourhoods_{city}"
    tgt = f"{CLEAN_DB}.airbnb_neighbourhoods_{city}"
    print(f"\n  {city}")

    try:
        df = spark.table(src)
        df = select_available(df, NEIGHBOURHOODS_COLUMNS)
        df = df.withColumn("neighbourhood", F.initcap(F.trim(F.col("neighbourhood"))))
        df = df.filter(F.col("neighbourhood").isNotNull())
        df = df.withColumn("_cleaned_at", F.current_timestamp())
        write_clean(df, tgt)
        clean_log.append({"section": "neighbourhoods", "city": city, "status": "ok", "rows": df.count()})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        clean_log.append({"section": "neighbourhoods", "city": city, "status": "error", "error": str(e)})

---
# Part E — Neighbourhoods GeoJSON

Passed through as raw text — GeoPandas parsing happens in the gold layer on a single-node step. Spark cannot run GeoPandas on distributed workers.

In [0]:
print("Passing through GeoJSON...")

for city in CITIES:
    src = f"{RAW_DB}.airbnb_neighbourhoods_geo_{city}"
    tgt = f"{CLEAN_DB}.airbnb_neighbourhoods_geo_{city}"
    print(f"\n  {city}")

    try:
        df = spark.table(src)
        df = select_available(df, GEOJSON_COLUMNS)
        df = df.withColumn("_cleaned_at", F.current_timestamp())
        write_clean(df, tgt)
        clean_log.append({"section": "neighbourhoods_geo", "city": city, "status": "ok", "rows": df.count()})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        clean_log.append({"section": "neighbourhoods_geo", "city": city, "status": "error", "error": str(e)})

---
# Part F — House Prices

**Raw schema:** 124 columns (4 identifier + 120 quarterly price columns)  
**Clean schema:** 6 columns  

| Transformation | Detail |
|---------------|--------|
| Column selection | Keep `msoa_code`, `msoa_name`, `local_authority_code`, `local_authority_name`, `year_ending_sep_2025`, `year_ending_sep_2015` only |
| Price columns | Strip commas, cast to float |
| `price_growth_10yr` | Calculated as `(sep_2025 - sep_2015) / sep_2015` |
| Null prices | Drop rows where most recent price is null |

> **Coverage:** England and Wales only — Edinburgh rows absent. Sep 2015 kept as baseline for 10-year price growth calculation.

In [0]:
print("Cleaning house prices...")
HP_CLEAN_TABLE = f"{CLEAN_DB}.house_prices_msoa"

try:
    df = spark.table(f"{RAW_DB}.house_prices_msoa")

    # Keep identifier columns and two price periods only
    df = df.select(
        "local_authority_code",
        "local_authority_name",
        "msoa_code",
        "msoa_name",
        F.col(HP_PERIOD_RECENT).alias("median_house_price_2025"),
        F.col(HP_PERIOD_BASELINE).alias("median_house_price_2015"),
        "_ingested_at",
    )

    # Strip commas and cast to float (raw values like "285,000")

    for col_alias in ["median_house_price_2025", "median_house_price_2015"]:
        df = df.withColumn(
        col_alias,
        F.when(F.col(col_alias).rlike("\\[.*\\]"), None)
         .when(F.col(col_alias).isNull(), None)
         .otherwise(F.regexp_replace(F.col(col_alias), ",", "").cast(FloatType()))
        )

    # Drop rows where most recent price is null
    df = df.filter(F.col("median_house_price_2025").isNotNull())

    # Calculate 10-year price growth
    df = df.withColumn(
        "price_growth_10yr",
        F.when(
            F.col("median_house_price_2015").isNotNull() & (F.col("median_house_price_2015") > 0),
            (F.col("median_house_price_2025") - F.col("median_house_price_2015")) / F.col("median_house_price_2015")
        ).otherwise(None).cast(FloatType())
    )

    df = df.withColumn("_cleaned_at", F.current_timestamp())
    write_clean(df, HP_CLEAN_TABLE)
    clean_log.append({"section": "house_prices", "city": "all", "status": "ok", "rows": df.count()})

except Exception as e:
    print(f"  ✗ {HP_CLEAN_TABLE} — {e}")
    clean_log.append({"section": "house_prices", "city": "all", "status": "error", "error": str(e)})

---
# Part G — Amenities

**GP surgeries and parks:** LAD level, England and Wales only  
**Rail stations:** MSOA level, England and Wales only  

| Transformation | Detail |
|---------------|--------|
| `[c]` suppressed values | Replaced with null |
| `[note X]` markers | Stripped from values |
| Count columns | Cast to float (some values contain decimals) |
| `lad_name` / `msoa_name` | `initcap` + `trim` for join consistency |

> Edinburgh has no rows in GP surgeries or parks — noted as a data gap in assumptions.

In [0]:
print("Cleaning GP surgeries...")
GP_CLEAN_TABLE = f"{CLEAN_DB}.amenities_gp_surgeries"

try:
    df = spark.table(f"{RAW_DB}.amenities_gp_surgeries")

    # Keep useful columns only
    df = df.select(
        "lad_code",
        F.initcap(F.trim(F.col("lad_name"))).alias("lad_name"),
        "count_note_3",
        "gps_per_100000_people",
        "_ingested_at",
    )

    # Replace suppressed [c] and note markers with null, cast to float
    for c in ["count_note_3", "gps_per_100000_people"]:
        df = df.withColumn(
            c,
            F.when(F.col(c).rlike("\\[.*\\]"), None)
             .otherwise(F.regexp_replace(F.col(c), ",", ""))
             .cast(FloatType())
        )

    # Rename for clarity
    df = df.withColumnRenamed("count_note_3", "gp_surgery_count")

    df = df.withColumn("_cleaned_at", F.current_timestamp())
    write_clean(df, GP_CLEAN_TABLE)
    clean_log.append({"section": "amenities_gp", "city": "all", "status": "ok", "rows": df.count()})

except Exception as e:
    print(f"  ✗ {GP_CLEAN_TABLE} — {e}")
    clean_log.append({"section": "amenities_gp", "city": "all", "status": "error", "error": str(e)})

In [0]:
print("Cleaning parks...")
PARKS_CLEAN_TABLE = f"{CLEAN_DB}.amenities_parks"

try:
    df = spark.table(f"{RAW_DB}.amenities_parks")

    df = df.select(
        "lad_code",
        F.initcap(F.trim(F.col("lad_name"))).alias("lad_name"),
        "total_parks_and_play_areas_count_note_3",
        "play_area",
        "playing_field",
        "public_park_or_garden",
        "recreation_ground",
        "parks_and_play_areas_per_100000_people",
        "_ingested_at",
    )

    # Replace suppressed values and cast to float
    count_cols = [
        "total_parks_and_play_areas_count_note_3",
        "play_area", "playing_field", "public_park_or_garden",
        "recreation_ground", "parks_and_play_areas_per_100000_people"
    ]
    for c in count_cols:
        df = df.withColumn(
            c,
            F.when(F.col(c).rlike("\\[.*\\]"), None)
             .otherwise(F.regexp_replace(F.col(c), ",", ""))
             .cast(FloatType())
        )

    # Rename for clarity
    df = df.withColumnRenamed("total_parks_and_play_areas_count_note_3", "total_parks_count")

    df = df.withColumn("_cleaned_at", F.current_timestamp())
    write_clean(df, PARKS_CLEAN_TABLE)
    clean_log.append({"section": "amenities_parks", "city": "all", "status": "ok", "rows": df.count()})

except Exception as e:
    print(f"  ✗ {PARKS_CLEAN_TABLE} — {e}")
    clean_log.append({"section": "amenities_parks", "city": "all", "status": "error", "error": str(e)})

In [0]:
print("Cleaning rail stations...")
RAIL_CLEAN_TABLE = f"{CLEAN_DB}.amenities_rail_stations"

try:
    df = spark.table(f"{RAW_DB}.amenities_rail_stations")

    df = df.select(
        "msoa_code",
        F.initcap(F.trim(F.col("msoa_name"))).alias("msoa_name"),
        "less_than_15_minute_walk",
        "less_than_30_minute_walk",
        "less_than_60_minute_walk",
        "_ingested_at",
    )

    # Replace suppressed values and cast to float (percentage values)
    for c in ["less_than_15_minute_walk", "less_than_30_minute_walk", "less_than_60_minute_walk"]:
        df = df.withColumn(
            c,
            F.when(F.col(c).rlike("\\[.*\\]"), None)
             .otherwise(F.col(c))
             .cast(FloatType())
        )

    df = df.withColumn("_cleaned_at", F.current_timestamp())
    write_clean(df, RAIL_CLEAN_TABLE)
    clean_log.append({"section": "amenities_rail", "city": "all", "status": "ok", "rows": df.count()})

except Exception as e:
    print(f"  ✗ {RAIL_CLEAN_TABLE} — {e}")
    clean_log.append({"section": "amenities_rail", "city": "all", "status": "error", "error": str(e)})

---
# Part H — Rent Data

**Raw schema:** 41 columns, 48,552 rows (monthly from Jan 2015)  
**Clean schema:** 8 columns, most recent 12 months only  

| Transformation | Detail |
|---------------|--------|
| Time filter | Most recent 12 months only |
| Column selection | Keep `area_code`, `area_name`, `time_period`, rental price columns only |
| Rental price columns | Strip `[note X]` markers, cast to float |
| Index/change columns | Dropped — not needed for STR vs LTR comparison |

> **Edinburgh not present** in this dataset — Scottish rental data must be sourced separately.

In [0]:
print("Cleaning rent data...")
RENT_CLEAN_TABLE = f"{CLEAN_DB}.rent_data"

try:
    df = spark.table(f"{RAW_DB}.rent_data")

    # Keep rental price columns and identifiers only
    df = df.select(
        "time_period",
        "area_code",
        "area_name",
        "region_or_country_name",
        "rental_price",
        "rental_price_one_bed",
        "rental_price_two_bed",
        "rental_price_three_bed",
        "rental_price_four_or_more_bed",
        "_ingested_at",
    )

    # time_period is already a timestamp — cast directly to date
    df = df.withColumn(
    "period_date",
    F.to_date(F.col("time_period"))
    )

    
    # Filter to most recent 12 months
    max_date = df.agg(F.max("period_date")).collect()[0][0]
    cutoff   = F.add_months(F.lit(max_date), -12)
    df = df.filter(F.col("period_date") >= cutoff)
    print(f"  Most recent period: {max_date} — keeping from {cutoff}")

    # Strip note markers e.g. '[note 1]' and cast rental prices to float
    price_cols = [
        "rental_price", "rental_price_one_bed", "rental_price_two_bed",
        "rental_price_three_bed", "rental_price_four_or_more_bed"
    ]
    for c in price_cols:
        df = df.withColumn(
            c,
            F.when(F.col(c).rlike("\\[.*\\]"), None)
             .otherwise(F.regexp_replace(F.col(c), ",", ""))
             .cast(FloatType())
        )

    df = df.withColumn("_cleaned_at", F.current_timestamp())
    write_clean(df, RENT_CLEAN_TABLE)
    clean_log.append({"section": "rent", "city": "all", "status": "ok", "rows": df.count()})

except Exception as e:
    print(f"  ✗ {RENT_CLEAN_TABLE} — {e}")
    clean_log.append({"section": "rent", "city": "all", "status": "error", "error": str(e)})

---
## Cleaning Summary

In [0]:
summary = pd.DataFrame(clean_log)
display(summary)

failures = summary[summary["status"] == "error"]
if not failures.empty:
    raise RuntimeError(
        f"Cleaning failed for:\n{failures[['section', 'city', 'error']].to_string()}"
    )

print("\nCleaning complete.")

---
## Spot Checks

In [0]:
# Listings: confirm price is float, outlier flags present
spark.sql("""
    SELECT id, neighbourhood_cleansed, room_type,
           price, price_is_outlier,
           minimum_nights, minimum_nights_is_outlier,
           review_scores_rating, host_is_superhost
    FROM airbnb_app.clean.airbnb_listings_edinburgh
    LIMIT 10
""").display()

In [0]:
# Listings: outlier counts and price range per city
spark.sql("""
    SELECT _city,
           COUNT(*)                                             AS total_listings,
           SUM(CASE WHEN price_is_outlier THEN 1 END)          AS price_outliers,
           SUM(CASE WHEN minimum_nights_is_outlier THEN 1 END) AS min_nights_outliers,
           ROUND(AVG(price), 2)                                AS avg_price,
           MIN(price)                                          AS min_price,
           MAX(price)                                          AS max_price
    FROM (
        SELECT * FROM airbnb_app.clean.airbnb_listings_london
        UNION ALL SELECT * FROM airbnb_app.clean.airbnb_listings_manchester
        UNION ALL SELECT * FROM airbnb_app.clean.airbnb_listings_edinburgh
        UNION ALL SELECT * FROM airbnb_app.clean.airbnb_listings_bristol
    )
    GROUP BY _city ORDER BY _city
""").display()

In [0]:
# Calendar: confirm available is boolean, price columns absent
spark.sql("""
    SELECT listing_id, date, available, minimum_nights
    FROM airbnb_app.clean.airbnb_calendar_edinburgh
    LIMIT 10
""").display()

In [0]:
# Reviews: confirm no null comments, comment_length present
spark.sql("""
    SELECT listing_id, date, comment_length, LEFT(comments, 150) AS comment_preview
    FROM airbnb_app.clean.airbnb_reviews_edinburgh
    ORDER BY comment_length DESC
    LIMIT 5
""").display()

In [0]:
# House prices: confirm two periods present and price growth calculated
spark.sql("""
    SELECT local_authority_name, msoa_name,
           median_house_price_2025, median_house_price_2015,
           ROUND(price_growth_10yr * 100, 1) AS price_growth_10yr_pct
    FROM airbnb_app.clean.house_prices_msoa
    WHERE local_authority_name LIKE '%Manchester%'
    LIMIT 10
""").display()

In [0]:
# Rent: confirm most recent 12 months, rental price columns present
spark.sql("""
    SELECT time_period, area_name, rental_price,
           rental_price_one_bed, rental_price_two_bed
    FROM airbnb_app.clean.rent_data
    WHERE area_name LIKE '%London%'
    ORDER BY period_date DESC
    LIMIT 10
""").display()

In [0]:
# Amenities: confirm suppressed values handled
spark.sql("""
    SELECT lad_name, gp_surgery_count, gps_per_100000_people
    FROM airbnb_app.clean.amenities_gp_surgeries
    LIMIT 10
""").display()

## Notes

**Listings**
- Outliers flagged not dropped — `price_is_outlier` and `minimum_nights_is_outlier` are boolean columns using IQR × 3. Exclude for scoring, keep for market analysis.
- `estimated_occupancy_l365d` and `estimated_revenue_l365d` are pre-calculated by Inside Airbnb — document as external estimates.

**Calendar**
- `available = false` cannot distinguish booked from host-blocked nights. State this assumption in any occupancy proxy.

**House prices**
- Sep 2025 = most recent price. Sep 2015 = baseline for 10-year growth.
- England and Wales only — Edinburgh absent.

**Amenities**
- `[c]` suppressed values replaced with null — affects some rural LADs.
- GP surgeries and parks: LAD level, Edinburgh absent.
- Rail stations: MSOA level, joins directly to listings via `msoa_code` in gold layer.

**Rent data**
- Edinburgh not present — Scottish rental data needed from `gov.scot`.
- Filtered to most recent 12 months — use median across months for a stable annual estimate.

**Next step:** Run `03_gold.ipynb` to join enrichment datasets, calculate occupancy proxy, build investment scores, and write to `airbnb_app.gold`.